In [1]:
import sys
import MySQLdb

print(sys.executable)
print(MySQLdb.version_info)

/Users/steve.tsao/projects-practice/python-course/.venv/bin/python
(2, 2, 8, 'final', 0)


In [7]:
from getpass import getpass

import MySQLdb
import MySQLdb.cursors

DB_PASSWORD = getpass("grade_app password: ")

DB_CONFIG = {
    "host": "127.0.0.1",
    "port": 3306,
    "user": "grade_app",
    "password": DB_PASSWORD,
    "database": "mydatabase",
    "charset": "utf8mb4",
    "cursorclass": MySQLdb.cursors.DictCursor,
}


def open_db():
    return MySQLdb.connect(**DB_CONFIG)

db = open_db()
cursor = db.cursor()

try:
    cursor.execute("SELECT VERSION() AS mysql_version")
    server_info = cursor.fetchone()
    print(server_info["mysql_version"])
finally:
    cursor.close()
    db.close()

grade_app password:  ········


8.4.11


In [9]:
student_rows = [
    ("S001", "isaac", "M", 60, 72, 32, 52, 86),
    ("S002", "amy", "F", 50, 22, 80, 15, 93),
]

insert_sql = """
    INSERT INTO students (
        student_no,
        name,
        gender_code,
        chinese,
        english,
        math,
        social_science,
        science
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

db = open_db()
cursor = db.cursor()

try:
    cursor.executemany(insert_sql, student_rows)
    db.commit()
except MySQLdb.Error as error:
    db.rollback()
    raise RuntimeError("新增學生失敗，這批資料已回滾") from error
finally:
    cursor.close()
    db.close()

RuntimeError: 新增學生失敗，這批資料已回滾

In [24]:
select_sql = """
    SELECT
        student_no,
        name,
        gender_code,
        chinese,
        english,
        math,
        social_science,
        science
    FROM students
    ORDER BY student_no
"""

db = open_db()
cursor = db.cursor()

try:
    cursor.execute(select_sql)
    students = cursor.fetchall()

    for student in students:
        print(
            student["student_no"],
            student["name"],
            student["chinese"],
            student["english"],
            student["math"],
            student["social_science"],
            student["science"],
        )
finally:
    cursor.close()
    db.close()

S001 isaac 60 72 32 52 86
S002 amy 50 22 80 15 93
S003 loe 70 48 69 50 66
S004 judy 89 90 59 28 39
S005 lily 37 43 67 34 43
S006 evan 12 57 97 100 100
S007 joyce 58 48 47 98 37
S008 yoshi 58 49 79 38 20
S009 amber 57 23 6 28 69
S010 alex 59 69 20 60 79
S011 sophia 79 70 70 49 70


In [23]:
import csv
from pathlib import Path

CSV_PATH = Path("2-exam_score.csv")

if not CSV_PATH.is_file():
    raise FileNotFoundError(f"找不到 CSV：{CSV_PATH}")

csv_rows = []

with CSV_PATH.open(encoding="utf-8-sig", newline="") as csv_file:
    reader = csv.DictReader(csv_file)

    for line_number, row in enumerate(reader, start=2):
        scores = [
            int(row["chinese"]),
            int(row["english"]),
            int(row["math"]),
            int(row["social_science"]),
            int(row["science"]),
        ]

        invalid_score_found = False

        for score in scores:
            if score < 0 or score > 100:
                invalid_score_found = True
                break

        if invalid_score_found:
            raise ValueError(f"CSV 第 {line_number} 列含有 0～100 以外的成績")

        csv_rows.append(
            (
                row["student_no"].strip(),
                row["name"].strip(),
                row["gender_code"].strip().upper() or None,
                *scores,
            )
        )

insert_sql = """
    INSERT IGNORE INTO students (
        student_no,
        name,
        gender_code,
        chinese,
        english,
        math,
        social_science,
        science
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

db = open_db()
cursor = db.cursor()

try:
    cursor.executemany(insert_sql, csv_rows)
    db.commit()
except MySQLdb.Error as error:
    db.rollback()
    raise RuntimeError("CSV 匯入失敗，整批資料已回滾") from error
finally:
    cursor.close()
    db.close()

In [26]:
update_sql = "UPDATE students SET english = %s WHERE student_no = %s"
update_values = (98, "S011")

db = open_db()
cursor = db.cursor()

try:
    cursor.execute(update_sql, update_values)
    affected_rows = cursor.rowcount

    cursor.execute(
        "SELECT english FROM students WHERE student_no = %s",
        ("S011",),
    )
    sophia = cursor.fetchone()

    if sophia is None:
        db.rollback()
        raise RuntimeError("找不到學號 S011，沒有任何資料可更新")

    db.commit()
    print(f"實際變更 {affected_rows} 列；目前英文成績為 {sophia['english']}")
except MySQLdb.Error as error:
    db.rollback()
    raise RuntimeError("更新 Sophia 成績失敗") from error
finally:
    cursor.close()
    db.close()

實際變更 1 列；目前英文成績為 98


In [27]:
average_sql = """
    SELECT
        ROUND(AVG(chinese), 2) AS chinese_avg,
        ROUND(AVG(english), 2) AS english_avg,
        ROUND(AVG(math), 2) AS math_avg,
        ROUND(AVG(social_science), 2) AS social_science_avg,
        ROUND(AVG(science), 2) AS science_avg
    FROM students
"""

db = open_db()
cursor = db.cursor()

try:
    cursor.execute(average_sql)
    averages = cursor.fetchone()

    if averages["chinese_avg"] is None:
        print("尚無成績")
    else:
        print(averages)
finally:
    cursor.close()
    db.close()

{'chinese_avg': Decimal('57.18'), 'english_avg': Decimal('56.27'), 'math_avg': Decimal('56.91'), 'social_science_avg': Decimal('50.18'), 'science_avg': Decimal('63.82')}
